# Hyperparameter Optimization

Hyperparameters are settings chosen before model fitting. Grid search tries predefined combinations on training folds. Keep the final holdout untouched until the selection procedure is complete.

**Synthetic educational data.**

In [ ]:
import pathlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

def locate(filename):
    paths=[pathlib.Path('data')/filename,pathlib.Path('../data')/filename,pathlib.Path(filename),pathlib.Path('/content')/filename]
    for path in paths:
        if path.exists(): return path
    raise FileNotFoundError(f'Upload {filename} or run notebook from repository root')

churn=pd.read_csv(locate('customer_churn_synthetic.csv'))
sales=pd.read_csv(locate('store_sales_synthetic.csv'))
segments=pd.read_csv(locate('customer_segments_synthetic.csv'))
print('Loaded:',len(churn),'churn rows,',len(sales),'sales rows,',len(segments),'segment rows')

In [ ]:
display(churn.head(3)); print('Class balance:'); print(churn['churn'].value_counts(normalize=True).round(3))

## Core demonstration
Run the model below, then interpret the output in business terms.

In [ ]:
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
X=churn.drop(columns=['customer_id','churn']); y=churn.churn
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=.2,stratify=y,random_state=42)
search=GridSearchCV(RandomForestClassifier(n_estimators=30,random_state=42),{'max_depth':[3,6,None],'min_samples_leaf':[1,5]},cv=3,scoring='roc_auc',n_jobs=1)
search.fit(X_train,y_train)
print('Best params:',search.best_params_)
print('Holdout ROC-AUC:',round(roc_auc_score(y_test,search.predict_proba(X_test)[:,1]),3))

## Visualization and interpretation
A simple distribution plot helps reveal the range and shape of an important feature.

In [ ]:
fig,ax=plt.subplots(figsize=(7,4))
churn['tenure_months'].hist(bins=25,ax=ax)
ax.set(title='Synthetic feature distribution',xlabel='Tenure feature',ylabel='Count')
plt.tight_layout(); plt.show()

## Questions
1. What baseline should this model beat?
2. What would be a costly error?
3. How could leakage enter this workflow?
4. What additional validation would you require before deployment?